# Classify Land Use Intensity and Calc Proportions per 1km grid cells


In [ ]:
import rasterio
import numpy as np
from rasterio.windows import Window
import gc
from pathlib import Path

# Base directory
base_path = Path("/home/georg/data/LEON_P5_BII")

## Load Data

### multiple in one variable

In [ ]:
# Base directory
BASE_DIR = base_path / "EO_data_prep"

# Define all file paths in a structured dictionary
FILES = {
    'dnk': {
        2023: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_dnk_2023.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2023.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2023.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2025_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2023.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif" 
        },
        2022: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_dnk_2022.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2022.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2022.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2022.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2021: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_dnk_2021.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2021.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2021.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2021.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2020: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_dnk_2020.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2020.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2020.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2020.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2019: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_dnk_2019.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2019.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2019.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        },
        2018: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_dnk_2018.tif",
            'eca': BASE_DIR / "ECA/eca_dnk_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_dnk_2018.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_dnk_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_dnk_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_dnk_2015_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_DNK_2018.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_dnk.tif"
        }
    },
    'nld': {
        2023: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_nld_2023.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2023.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2023.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2025_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2023.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2022: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_nld_2022.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2023.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2022.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2022.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2022.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2021: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_nld_2021.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2021.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2021.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2021.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2020: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_nld_2020.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2021.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2020.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2021.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2020.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2020.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2019: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_nld_2019.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2019.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2020_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2019.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        },
        2018: {
            'lc': BASE_DIR / "LC_urban_pasture/lc_urban5_pasture_nld_2018.tif",
            'eca': BASE_DIR / "ECA/eca_nld_2018.tif",
            'rsd': BASE_DIR / "Grazing_intensity/RSD_nld_2018.tif",
            'swf': BASE_DIR / "Small_Woody_Features/swf_density_500m_gaussian_nld_2018.tif",
            'bare': BASE_DIR / "Bare_Total/bare_total_nld_2018.tif",
            'ghsl': BASE_DIR / "GHSL/ghsl_pop_nld_2015_clip.tif",
            'nt': BASE_DIR / "Nightlight/VIIRS_NTL_NLD_2018.tif",
            'sv': BASE_DIR / "Structural_Diversity/cv_height_100m_nld.tif"
        }
    }
}

# access files like:
# FILES['dnk'][2023]['lc']
# FILES['nld'][2021]['eca']

## Single Year Land Use Intensity

In [ ]:
# # ============================================
# # Thresholds - ADJUST HERE
# # ============================================
# THRESHOLDS = {
#     'sv': {'low': 33, 'high': 66},
#     'plantation': {'low': 33, 'high': 66},
#     'crop': {
#         'bare_low': 60,   # days
#         'bare_high': 90,  # days
#         'swf_high': 1000  # for minimal intensity
#     },
#     'pasture': {'low': 0.2, 'high': 0.65},
#     'urban': {
#         'nt_thresh1': 2,    # nightlight threshold 1
#         'nt_thresh2': 10,   # nightlight threshold 2
#         'nt_thresh3': 20,   # nightlight threshold 3
#         'ghsl_thresh1': 500,   # settlement threshold 1
#         'ghsl_thresh2': 3000   # settlement threshold 2
#     }
# }

# # Classes to set as No Data
# NO_DATA_CLASSES = [8, 10, 11, 253, 254, 255]
# NO_DATA_VALUE = 255

# # ============================================
# # Process Country Year
# # ============================================
# year, country = 2023, 'dnk'
# lc_file = FILES[country][year]['lc']
# eca_file = FILES[country][year]['eca']
# bare_file = FILES[country][year]['bare']
# ghsl_file = FILES[country][year]['ghsl']
# swf_file = FILES[country][year]['swf']
# rsd_file = FILES[country][year]['rsd']
# nt_file = FILES[country][year]['nt']

# print(f"\n{'='*60}")
# print(f"Processing {country.upper()} {year}")
# print('='*60)

# # Load auxiliary layers
# print("\nLoading auxiliary layers...")
# with rasterio.open(eca_file) as src:
#     eca = src.read(1)
# with rasterio.open(bare_file) as src:
#     bare = src.read(1)
# with rasterio.open(ghsl_file) as src:
#     ghsl = src.read(1)
# with rasterio.open(swf_file) as src:
#     swf = src.read(1)
# with rasterio.open(rsd_file) as src:
#     rsd = src.read(1)
#     rsd = np.nan_to_num(rsd, nan=0.0)
# with rasterio.open(nt_file) as src:
#     nt = src.read(1)

# # ============================================
# # CHECK AUXILIARY LAYER DISTRIBUTIONS
# # ============================================
# print(f"\nAUXILIARY LAYER DISTRIBUTIONS (using current thresholds):")

# # ECA (for SV & Plantation)
# t = THRESHOLDS['sv']
# print(f"\n  ECA (for SV & Plantation):")
# print(f"  Min: {eca.min():.1f}, Max: {eca.max():.1f}, Mean: {eca.mean():.1f}")
# print(f"  <{t['low']} (intense): {np.sum(eca < t['low']) / eca.size * 100:.1f}%")
# print(f"  {t['low']}-{t['high']} (light): {np.sum((eca >= t['low']) & (eca <= t['high'])) / eca.size * 100:.1f}%")
# print(f"  >{t['high']} (minimal): {np.sum(eca > t['high']) / eca.size * 100:.1f}%")

# # Bare (for Crop)
# t = THRESHOLDS['crop']
# print(f"\n  Bare (for Crop):")
# print(f"  Min: {bare.min():.1f}, Max: {bare.max():.1f}, Mean: {bare.mean():.1f}")
# print(f"  <{t['bare_low']} days: {np.sum(bare < t['bare_low']) / bare.size * 100:.1f}%")
# print(f"  {t['bare_low']}-{t['bare_high']} days: {np.sum((bare >= t['bare_low']) & (bare < t['bare_high'])) / bare.size * 100:.1f}%")
# print(f"  >{t['bare_high']} days: {np.sum(bare >= t['bare_high']) / bare.size * 100:.1f}%")

# # SWF (for Crop)
# print(f"\n  SWF (for Crop):")
# print(f"  Min: {swf.min():.1f}, Max: {swf.max():.1f}, Mean: {swf.mean():.1f}")
# print(f"  <{t['swf_high']}: {np.sum(swf < t['swf_high']) / swf.size * 100:.1f}%")
# print(f"  >={t['swf_high']}: {np.sum(swf >= t['swf_high']) / swf.size * 100:.1f}%")

# # GHSL & Nightlight (for Urban)
# t = THRESHOLDS['urban']
# print(f"\n  GHSL (for Urban):")
# print(f"  Min: {ghsl.min():.1f}, Max: {ghsl.max():.1f}, Mean: {ghsl.mean():.1f}")
# print(f"  Valid (>=0): {np.sum(ghsl >= 0) / ghsl.size * 100:.1f}%")
# print(f"  <{t['ghsl_thresh1']}: {np.sum((ghsl >= 0) & (ghsl < t['ghsl_thresh1'])) / ghsl.size * 100:.1f}%")
# print(f"  {t['ghsl_thresh1']}-{t['ghsl_thresh2']}: {np.sum((ghsl >= t['ghsl_thresh1']) & (ghsl < t['ghsl_thresh2'])) / ghsl.size * 100:.1f}%")
# print(f"  >={t['ghsl_thresh2']}: {np.sum(ghsl >= t['ghsl_thresh2']) / ghsl.size * 100:.1f}%")

# print(f"\n  Nightlight (for Urban):")
# print(f"  Min: {nt.min():.2f}, Max: {nt.max():.2f}, Mean: {nt.mean():.2f}")
# print(f"  <{t['nt_thresh1']}: {np.sum(nt < t['nt_thresh1']) / nt.size * 100:.1f}%")
# print(f"  {t['nt_thresh1']}-{t['nt_thresh2']}: {np.sum((nt >= t['nt_thresh1']) & (nt < t['nt_thresh2'])) / nt.size * 100:.1f}%")
# print(f"  {t['nt_thresh2']}-{t['nt_thresh3']}: {np.sum((nt >= t['nt_thresh2']) & (nt < t['nt_thresh3'])) / nt.size * 100:.1f}%")
# print(f"  >={t['nt_thresh3']}: {np.sum(nt >= t['nt_thresh3']) / nt.size * 100:.1f}%")

# # RSD (for Pasture)
# t = THRESHOLDS['pasture']
# print(f"\n  RSD (for Pasture):")
# print(f"  Min: {rsd.min():.3f}, Max: {rsd.max():.3f}, Mean: {rsd.mean():.3f}")
# print(f"  <{t['low']} (minimal): {np.sum(rsd < t['low']) / rsd.size * 100:.1f}%")
# print(f"  {t['low']}-{t['high']} (light): {np.sum((rsd >= t['low']) & (rsd <= t['high'])) / rsd.size * 100:.1f}%")
# print(f"  >{t['high']} (intense): {np.sum(rsd > t['high']) / rsd.size * 100:.1f}%")

# # ============================================
# # Process LC in chunks
# # ============================================
# print(f"\n{'='*60}")
# print("Starting classification...")

# with rasterio.open(lc_file) as lc_src:
#     profile = lc_src.profile.copy()
#     profile.update(nodata=NO_DATA_VALUE)
#     height, width = lc_src.shape
    
#     # Calculate scale factors
#     eca_scale_row = height / eca.shape[0]
#     eca_scale_col = width / eca.shape[1]
#     bare_scale_row = height / bare.shape[0]
#     bare_scale_col = width / bare.shape[1]
#     ghsl_scale_row = height / ghsl.shape[0]
#     ghsl_scale_col = width / ghsl.shape[1]
#     swf_scale_row = height / swf.shape[0]
#     swf_scale_col = width / swf.shape[1]
#     rsd_scale_row = height / rsd.shape[0]
#     rsd_scale_col = width / rsd.shape[1]
#     nt_scale_row = height / nt.shape[0]
#     nt_scale_col = width / nt.shape[1]

#     print(f"\nResolution scale factors:")
#     print(f"  ECA: {eca_scale_row:.1f}x, Bare: {bare_scale_row:.1f}x, GHSL: {ghsl_scale_row:.1f}x")
#     print(f"  SWF: {swf_scale_row:.1f}x, RSD: {rsd_scale_row:.1f}x, Nightlight: {nt_scale_row:.1f}x")

#     # Create output array
#     intensity = np.zeros((height, width), dtype=profile['dtype'])
    
#     CHUNK_SIZE = 2000
    
#     for row_start in range(0, height, CHUNK_SIZE):
#         row_end = min(row_start + CHUNK_SIZE, height)
#         print(f"  Processing rows {row_start}-{row_end}/{height}...")
        
#         window = Window(0, row_start, width, row_end - row_start)
#         lc = lc_src.read(1, window=window)
#         chunk_intensity = lc.copy()
        
#         # --- Set No Data classes ---
#         no_data_mask = np.isin(lc, NO_DATA_CLASSES)
#         chunk_intensity[no_data_mask] = NO_DATA_VALUE
        
#         # --- SV ---
#         sv_mask = np.isin(lc, [21, 22, 23, 24])
#         sv_rows, sv_cols = np.where(sv_mask)
#         if len(sv_rows) > 0:
#             global_rows = sv_rows + row_start
#             eca_rows = np.clip((global_rows / eca_scale_row).astype(int), 0, eca.shape[0] - 1)
#             eca_cols = np.clip((sv_cols / eca_scale_col).astype(int), 0, eca.shape[1] - 1)
#             eca_values = eca[eca_rows, eca_cols]
#             sv_classes = lc[sv_rows, sv_cols]
#             t = THRESHOLDS['sv']
#             intensity_class = np.where(eca_values < t['low'], 0,
#                                        np.where(eca_values <= t['high'], 1, 2))
#             chunk_intensity[sv_rows, sv_cols] = sv_classes * 10 + intensity_class
        
#         # --- Plantation ---
#         pl_mask = (lc == 3)
#         pl_rows, pl_cols = np.where(pl_mask)
#         if len(pl_rows) > 0:
#             global_rows = pl_rows + row_start
#             eca_rows = np.clip((global_rows / eca_scale_row).astype(int), 0, eca.shape[0] - 1)
#             eca_cols = np.clip((pl_cols / eca_scale_col).astype(int), 0, eca.shape[1] - 1)
#             eca_values = eca[eca_rows, eca_cols]
#             t = THRESHOLDS['plantation']
#             intensity_class = np.where(eca_values < t['low'], 0,
#                                        np.where(eca_values <= t['high'], 1, 2))
#             chunk_intensity[pl_rows, pl_cols] = 30 + intensity_class
        
#         # --- Crop (LC classes 6 and 7) ---
#         crop_mask = np.isin(lc, [6, 7])
#         crop_rows, crop_cols = np.where(crop_mask)
#         if len(crop_rows) > 0:
#             global_rows = crop_rows + row_start
            
#             bare_rows = np.clip((global_rows / bare_scale_row).astype(int), 0, bare.shape[0] - 1)
#             bare_cols = np.clip((crop_cols / bare_scale_col).astype(int), 0, bare.shape[1] - 1)
#             bare_values = bare[bare_rows, bare_cols]
            
#             swf_rows = np.clip((global_rows / swf_scale_row).astype(int), 0, swf.shape[0] - 1)
#             swf_cols = np.clip((crop_cols / swf_scale_col).astype(int), 0, swf.shape[1] - 1)
#             swf_values = swf[swf_rows, swf_cols]
            
#             t = THRESHOLDS['crop']
            
#             minimal = (bare_values < t['bare_low']) & (swf_values > t['swf_high'])
#             light = ((bare_values < t['bare_low']) & (swf_values <= t['swf_high'])) | \
#                     ((bare_values < t['bare_high']) & (swf_values > t['swf_high']))
#             intense = ((bare_values >= t['bare_low']) & (swf_values < t['swf_high'])) | \
#                       (bare_values >= t['bare_high'])
            
#             intensity_class = np.where(minimal, 0,
#                                       np.where(light, 1,
#                                               np.where(intense, 2, 1)))
            
#             chunk_intensity[crop_rows, crop_cols] = 70 + intensity_class
        
#         # --- Pasture (LC class 41) ---
#         pasture_mask = (lc == 41)
#         pasture_rows, pasture_cols = np.where(pasture_mask)
#         if len(pasture_rows) > 0:
#             global_rows = pasture_rows + row_start
#             rsd_rows = np.clip((global_rows / rsd_scale_row).astype(int), 0, rsd.shape[0] - 1)
#             rsd_cols = np.clip((pasture_cols / rsd_scale_col).astype(int), 0, rsd.shape[1] - 1)
#             rsd_values = rsd[rsd_rows, rsd_cols]
#             t = THRESHOLDS['pasture']
#             intensity_class = np.where(rsd_values < t['low'], 0,
#                                        np.where(rsd_values <= t['high'], 1, 2))
#             chunk_intensity[pasture_rows, pasture_cols] = 60 + intensity_class
        
#         # --- Urban (LC classes 1, 9) ---
#         urban_mask = np.isin(lc, [1, 9])
#         urban_rows, urban_cols = np.where(urban_mask)
#         if len(urban_rows) > 0:
#             global_rows = urban_rows + row_start
            
#             nt_rows = np.clip((global_rows / nt_scale_row).astype(int), 0, nt.shape[0] - 1)
#             nt_cols = np.clip((urban_cols / nt_scale_col).astype(int), 0, nt.shape[1] - 1)
#             nt_values = nt[nt_rows, nt_cols]
            
#             ghsl_rows = np.clip((global_rows / ghsl_scale_row).astype(int), 0, ghsl.shape[0] - 1)
#             ghsl_cols = np.clip((urban_cols / ghsl_scale_col).astype(int), 0, ghsl.shape[1] - 1)
#             ghsl_values = ghsl[ghsl_rows, ghsl_cols]
            
#             t = THRESHOLDS['urban']
            
#             nt_class = np.where(nt_values < t['nt_thresh1'], 4,
#                                np.where(nt_values < t['nt_thresh2'], 3,
#                                        np.where(nt_values < t['nt_thresh3'], 2, 1)))
            
#             intensity_class = np.where(
#                 (nt_class > 2) & (ghsl_values < t['ghsl_thresh1']), 0,
#                 np.where(
#                     (nt_class > 2) & (ghsl_values >= t['ghsl_thresh1']), 1,
#                     np.where(
#                         (nt_class == 2) & (ghsl_values < t['ghsl_thresh2']), 1,
#                         np.where(
#                             (nt_class == 2) & (ghsl_values >= t['ghsl_thresh2']), 2,
#                             np.where(nt_class == 1, 2, 0)
#                         )
#                     )
#                 )
#             )
            
#             chunk_intensity[urban_rows, urban_cols] = 10 + intensity_class
        
#         # --- Urban minimal (LC class 31) ---
#         urban_min_mask = (lc == 31)
#         urban_min_rows, urban_min_cols = np.where(urban_min_mask)
#         if len(urban_min_rows) > 0:
#             chunk_intensity[urban_min_rows, urban_min_cols] = 10
        
#         # Store chunk
#         intensity[row_start:row_end, :] = chunk_intensity
        
#         del lc, chunk_intensity
#         gc.collect()

# # Store result
# intensity_results = {
#     f"{country}_{year}": {
#         'data': intensity,
#         'profile': profile
#     }
# }

# # ============================================
# # Statistics
# # ============================================
# print(f"\n{'='*60}")
# print(f"Land Use Intensity Statistics for {country.upper()} {year}:")
# print(f"\nCurrent thresholds:")
# print(f"  Urban (LC 1,9,31): nightlight <{THRESHOLDS['urban']['nt_thresh1']}, {THRESHOLDS['urban']['nt_thresh1']}-{THRESHOLDS['urban']['nt_thresh2']}, {THRESHOLDS['urban']['nt_thresh2']}-{THRESHOLDS['urban']['nt_thresh3']}, >={THRESHOLDS['urban']['nt_thresh3']}")
# print(f"                     settlement <{THRESHOLDS['urban']['ghsl_thresh1']}, {THRESHOLDS['urban']['ghsl_thresh1']}-{THRESHOLDS['urban']['ghsl_thresh2']}, >={THRESHOLDS['urban']['ghsl_thresh2']}")
# print(f"  Crop (LC 6,7): minimal (Bare<{THRESHOLDS['crop']['bare_low']} & SWF>{THRESHOLDS['crop']['swf_high']}),")
# print(f"                 light ((Bare<{THRESHOLDS['crop']['bare_low']} & SWF<={THRESHOLDS['crop']['swf_high']}) OR (Bare<{THRESHOLDS['crop']['bare_high']} & SWF>{THRESHOLDS['crop']['swf_high']})),")
# print(f"                 intense ((Bare>={THRESHOLDS['crop']['bare_low']} & SWF<{THRESHOLDS['crop']['swf_high']}) OR Bare>={THRESHOLDS['crop']['bare_high']})")
# print(f"  Pasture (LC 41): minimal (RSD<{THRESHOLDS['pasture']['low']}), light (RSD {THRESHOLDS['pasture']['low']}-{THRESHOLDS['pasture']['high']}), intense (RSD>{THRESHOLDS['pasture']['high']})")
# print(f"\n{'Class':<35} {'Pixels':>15} {'%':>8}")
# print(f"{'-'*35} {'-'*15} {'-'*8}")

# total_pixels = intensity.size

# class_stats = {
#     10: 'Urban minimal', 11: 'Urban light', 12: 'Urban intense',
#     30: 'Plantation minimal', 31: 'Plantation light', 32: 'Plantation intense',
#     60: 'Pasture minimal', 61: 'Pasture light', 62: 'Pasture intense',
#     70: 'Crop minimal', 71: 'Crop light', 72: 'Crop intense',
#     210: 'SV Mature minimal', 211: 'SV Mature light', 212: 'SV Mature intense',
#     220: 'SV Intermediate minimal', 221: 'SV Intermediate light', 222: 'SV Intermediate intense',
#     230: 'SV Young minimal', 231: 'SV Young light', 232: 'SV Young intense',
#     240: 'SV Indeterminate minimal', 241: 'SV Indeterminate light', 242: 'SV Indeterminate intense',
# }

# for class_val, class_name in class_stats.items():
#     count = np.sum(intensity == class_val)
#     proportion = count / total_pixels * 100
#     if count > 0:
#         print(f"{class_name:<35} {count:>15,} {proportion:>7.3f}%")

# # No Data
# no_data_count = np.sum(intensity == NO_DATA_VALUE)
# print(f"{'No Data':<35} {no_data_count:>15,} {no_data_count/total_pixels*100:>7.3f}%")

# # Summary
# total_classified = np.sum(np.isin(intensity, list(class_stats.keys())))
# print(f"\n{'='*60}")
# print(f"Total classified: {total_classified:,} ({total_classified/total_pixels*100:.2f}%)")
# print(f"No Data: {no_data_count:,} ({no_data_count/total_pixels*100:.2f}%)")
# print(f"Other/unclassified: {total_pixels - total_classified - no_data_count:,} ({(total_pixels - total_classified - no_data_count)/total_pixels*100:.2f}%)")

# # Cleanup
# del eca, bare, ghsl, swf, rsd, nt
# gc.collect()

# print(f"\n✅ Done! Results in 'intensity_results' dict")

In [ ]:
# Save #
# Choose what to save
country = 'dnk'  # 'dnk' or 'nld'
year = 2023      # 2018, 2021, 2023
version = 'v1'       # version 

output_dir = Path(F"~/data/LEON_P5_BII/BII_LU_layer/Land_use_map/{version}").expanduser()
output_dir.mkdir(parents=True, exist_ok=True)

# Save 
key = f"{country}_{year}"
if key in intensity_results:
    output_file = output_dir / f"LU_map_{country}_{year}_{version}.tif"
    
    profile = intensity_results[key]['profile'].copy()
    profile.update(compress='lzw')
    
    with rasterio.open(output_file, 'w', **profile) as dst:
        dst.write(intensity_results[key]['data'], 1)
    
    print(f"✅ Saved: {output_file}")
else:
    print(f"❌ {key} not found in results")

## Batch process Land Use Intensity

In [ ]:
import numpy as np
import rasterio
from rasterio.windows import Window
import gc
from pathlib import Path

# Assumes base_path and FILES are already defined

##___Configuration___##
version = 'v1'  
countries = ['dnk', 'nld']
years = [2018, 2019, 2020, 2021, 2022, 2023]

# Paths
OUTPUT_DIR = base_path / "BII_LU_layer" / "Land_use_map" / version
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Thresholds
THRESHOLDS = {
    'sv': {'low': 15, 'high': 40}, # fragmentation thresholds
    'plantation': {'low': 0.2, 'high': 0.4},  # structural diversity thresholds
    'crop': {'bare_low': 60, 'bare_high': 90, 'swf_high': 1000}, # bare soil days and small woody features thresholds
    'pasture': {'low': 0.2, 'high': 0.65}, # RSD thresholds 
    'urban': {'nt_thresh1': 2, 'nt_thresh2': 10, 'nt_thresh3': 20, 
              'ghsl_thresh1': 10, 'ghsl_thresh2': 100} # nightlight and population thresholds
}

# Classes to set as No Data
NO_DATA_CLASSES = [8, 10, 11, 253, 254, 255]
NO_DATA_VALUE = 255

CHUNK_SIZE = 2000

# ============================================
# Helper Functions
# ============================================
def load_auxiliary_layers(files):
    """Load all auxiliary layers for a given year/country"""
    with rasterio.open(files['eca']) as src:
        eca = src.read(1)
    with rasterio.open(files['bare']) as src:
        bare = src.read(1)
    with rasterio.open(files['ghsl']) as src:
        ghsl = src.read(1)
    with rasterio.open(files['swf']) as src:
        swf = src.read(1)
    with rasterio.open(files['rsd']) as src:
        rsd = np.nan_to_num(src.read(1), nan=0.0)
    with rasterio.open(files['nt']) as src:
        nt = src.read(1)
    with rasterio.open(files['sv']) as src:
        sv = src.read(1)
    return eca, bare, ghsl, swf, rsd, nt, sv

def calculate_scale_factors(height, width, layers):
    """Calculate scale factors for all auxiliary layers"""
    return {
        'eca': (height / layers[0].shape[0], width / layers[0].shape[1]),
        'bare': (height / layers[1].shape[0], width / layers[1].shape[1]),
        'ghsl': (height / layers[2].shape[0], width / layers[2].shape[1]),
        'swf': (height / layers[3].shape[0], width / layers[3].shape[1]),
        'rsd': (height / layers[4].shape[0], width / layers[4].shape[1]),
        'nt': (height / layers[5].shape[0], width / layers[5].shape[1]),
        'sv': (height / layers[6].shape[0], width / layers[6].shape[1])
    }

def process_chunk(lc, row_start, layers, scales, thresholds):
    """Process a single chunk of land cover data"""
    eca, bare, ghsl, swf, rsd, nt, sv = layers
    chunk_intensity = lc.copy()
    
    # Set No Data classes first
    no_data_mask = np.isin(lc, NO_DATA_CLASSES)
    chunk_intensity[no_data_mask] = NO_DATA_VALUE
    
    # SV (Semi-Natural Vegetation)
    sv_mask = np.isin(lc, [21, 22, 23, 24])
    if np.any(sv_mask):
        rows, cols = np.where(sv_mask)
        global_rows = rows + row_start
        eca_rows = np.clip((global_rows / scales['eca'][0]).astype(int), 0, eca.shape[0] - 1)
        eca_cols = np.clip((cols / scales['eca'][1]).astype(int), 0, eca.shape[1] - 1)
        eca_vals = eca[eca_rows, eca_cols]
        t = thresholds['sv']
        intensity = np.where(eca_vals > t['high'], 0, np.where(eca_vals >= t['low'], 1, 2))
        chunk_intensity[rows, cols] = lc[rows, cols] * 10 + intensity
    
    # Plantation
    pl_mask = (lc == 3)
    if np.any(pl_mask):
        rows, cols = np.where(pl_mask)
        global_rows = rows + row_start
        sv_rows = np.clip((global_rows / scales['sv'][0]).astype(int), 0, sv.shape[0] - 1)
        sv_cols = np.clip((cols / scales['sv'][1]).astype(int), 0, sv.shape[1] - 1)
        sv_vals = sv[sv_rows, sv_cols]
        t = thresholds['plantation']
        intensity = np.where(sv_vals < t['low'], 2, np.where(sv_vals <= t['high'], 1, 0))
        chunk_intensity[rows, cols] = 30 + intensity
    
    # Crop
    crop_mask = np.isin(lc, [6, 7])
    if np.any(crop_mask):
        rows, cols = np.where(crop_mask)
        global_rows = rows + row_start
        bare_rows = np.clip((global_rows / scales['bare'][0]).astype(int), 0, bare.shape[0] - 1)
        bare_cols = np.clip((cols / scales['bare'][1]).astype(int), 0, bare.shape[1] - 1)
        bare_vals = bare[bare_rows, bare_cols]
        swf_rows = np.clip((global_rows / scales['swf'][0]).astype(int), 0, swf.shape[0] - 1)
        swf_cols = np.clip((cols / scales['swf'][1]).astype(int), 0, swf.shape[1] - 1)
        swf_vals = swf[swf_rows, swf_cols]
        t = thresholds['crop']
        minimal = (bare_vals < t['bare_low']) & (swf_vals > t['swf_high'])
        light = ((bare_vals < t['bare_low']) & (swf_vals <= t['swf_high'])) | \
                ((bare_vals < t['bare_high']) & (swf_vals > t['swf_high']))
        intense = ((bare_vals >= t['bare_low']) & (swf_vals < t['swf_high'])) | (bare_vals >= t['bare_high'])
        intensity = np.where(minimal, 0, np.where(light, 1, 2))
        chunk_intensity[rows, cols] = 70 + intensity
    
    # Pasture
    pasture_mask = (lc == 41)
    if np.any(pasture_mask):
        rows, cols = np.where(pasture_mask)
        global_rows = rows + row_start
        rsd_rows = np.clip((global_rows / scales['rsd'][0]).astype(int), 0, rsd.shape[0] - 1)
        rsd_cols = np.clip((cols / scales['rsd'][1]).astype(int), 0, rsd.shape[1] - 1)
        rsd_vals = rsd[rsd_rows, rsd_cols]
        t = thresholds['pasture']
        intensity = np.where(rsd_vals < t['low'], 0, np.where(rsd_vals <= t['high'], 1, 2))
        chunk_intensity[rows, cols] = 60 + intensity
    
    # Urban
    urban_mask = np.isin(lc, [1, 9])
    if np.any(urban_mask):
        rows, cols = np.where(urban_mask)
        global_rows = rows + row_start
        nt_rows = np.clip((global_rows / scales['nt'][0]).astype(int), 0, nt.shape[0] - 1)
        nt_cols = np.clip((cols / scales['nt'][1]).astype(int), 0, nt.shape[1] - 1)
        nt_vals = nt[nt_rows, nt_cols]
        ghsl_rows = np.clip((global_rows / scales['ghsl'][0]).astype(int), 0, ghsl.shape[0] - 1)
        ghsl_cols = np.clip((cols / scales['ghsl'][1]).astype(int), 0, ghsl.shape[1] - 1)
        ghsl_vals = ghsl[ghsl_rows, ghsl_cols]
        t = thresholds['urban']
        nt_class = np.where(nt_vals < t['nt_thresh1'], 4,
                           np.where(nt_vals < t['nt_thresh2'], 3,
                                   np.where(nt_vals < t['nt_thresh3'], 2, 1)))
        intensity = np.where((nt_class > 2) & (ghsl_vals < t['ghsl_thresh1']), 0,
                    np.where((nt_class > 2) & (ghsl_vals >= t['ghsl_thresh1']), 1,
                    np.where((nt_class == 2) & (ghsl_vals < t['ghsl_thresh2']), 1,
                    np.where((nt_class == 2) & (ghsl_vals >= t['ghsl_thresh2']), 2,
                    np.where(nt_class == 1, 2, 0)))))
        chunk_intensity[rows, cols] = 10 + intensity
    
    # Urban minimal
    if np.any(lc == 31):
        chunk_intensity[lc == 31] = 10
    
    return chunk_intensity

# ============================================
# Main Processing Loop
# ============================================
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

for country in countries:
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing {country.upper()} {year}")
        print('='*60)
        
        # Get file paths from already-loaded FILES dictionary
        file_paths = FILES[country][year]
        
        # Load auxiliary layers
        print("Loading auxiliary layers...")
        layers = load_auxiliary_layers(file_paths)
        
        # Process
        print("Starting classification...")
        with rasterio.open(file_paths['lc']) as lc_src:
            profile = lc_src.profile.copy()
            profile.update(compress='lzw', nodata=NO_DATA_VALUE)
            height, width = lc_src.shape
            
            scales = calculate_scale_factors(height, width, layers)
            print(f"Scale factors: ECA={scales['eca'][0]:.1f}x, RSD={scales['rsd'][0]:.1f}x, NT={scales['nt'][0]:.1f}x, SV={scales['sv'][0]:.1f}x")
            
            output_file = OUTPUT_DIR / f"lu_intensity_{country}_{year}_{version}.tif"
            
            with rasterio.open(output_file, 'w', **profile) as dst:
                for row_start in range(0, height, CHUNK_SIZE):
                    row_end = min(row_start + CHUNK_SIZE, height)
                    print(f"  Rows {row_start}-{row_end}/{height}")
                    
                    window = Window(0, row_start, width, row_end - row_start)
                    lc = lc_src.read(1, window=window)
                    
                    chunk_intensity = process_chunk(lc, row_start, layers, scales, THRESHOLDS)
                    dst.write(chunk_intensity, 1, window=window)
                    
                    del lc, chunk_intensity
                    gc.collect()
        
        print(f"✅ Saved: {output_file}")
        del layers
        gc.collect()

print("\n🎯 All processing complete!")

## Create 1km Grid cell with proportions

### Multiple LU files

In [ ]:
##___Configuration___##
version = 'v1'
countries = ['dnk', 'nld']
years = [2018, 2021, 2023]

# Directories
input_dir = base_path / "BII_LU_layer" / "Land_use_map" / version
output_dir = base_path / "BII_LU_layer" / "Land_use_proportions" / version
output_dir.mkdir(parents=True, exist_ok=True)


# ============================================
# Define intensity classes
# ============================================
INTENSITY_CLASSES = {
    'urban_minimal': 10,
    'urban_light': 11,
    'urban_intense': 12,
    'plantation_minimal': 30,
    'plantation_light': 31,
    'plantation_intense': 32,
    'pasture_minimal': 60,
    'pasture_light': 61,
    'pasture_intense': 62,
    'crop_minimal': 70,
    'crop_light': 71,
    'crop_intense': 72,
    'sv_mature_minimal': 210,
    'sv_mature_light': 211,
    'sv_mature_intense': 212,
    'sv_intermediate_minimal': 220,
    'sv_intermediate_light': 221,
    'sv_intermediate_intense': 222,
    'sv_young_minimal': 230,
    'sv_young_light': 231,
    'sv_young_intense': 232,
    'sv_indeterminate_minimal': 240,
    'sv_indeterminate_light': 241,
    'sv_indeterminate_intense': 242,
}


# ============================================
# Process all countries and years
# ============================================
for country in countries:
    for year in years:
        print(f"\n{'='*60}")
        print(f"Processing {country.upper()} {year}")
        print('='*60)
        
        # Load intensity file
        input_file = input_dir / f"lu_intensity_{country}_{year}_{version}.tif"
        
        if not input_file.exists():
            print(f"⚠️  File not found: {input_file}")
            continue
        
        print(f"Loading: {input_file.name}")
        
        with rasterio.open(input_file) as src:
            intensity = src.read(1)
            height, width = src.shape
            
            # 1km grid dimensions (100 pixels = 1km at 10m resolution)
            grid_height = height // 100
            grid_width = width // 100
            
            print(f"Original: {height} x {width} (10m)")
            print(f"Grid: {grid_height} x {grid_width} (1km)")
            
            # Store profile for output
            transform = src.transform * src.transform.scale(100, 100)
            output_profile = src.profile.copy()
            output_profile.update({
                'height': grid_height,
                'width': grid_width,
                'transform': transform,
                'dtype': 'float32',
                'count': len(INTENSITY_CLASSES),
                'compress': 'lzw',
                'nodata': -9999
            })
            
            # Calculate proportions for all classes
            print("Calculating proportions...")
            proportion_grids = {}
            
            for class_name, class_value in INTENSITY_CLASSES.items():
                print(f"  {class_name}...", end='')
                
                grid = np.zeros((grid_height, grid_width), dtype=np.float32)
                
                # Aggregate to 1km
                for i in range(grid_height):
                    for j in range(grid_width):
                        row_start = i * 100
                        col_start = j * 100
                        row_end = min(row_start + 100, height)
                        col_end = min(col_start + 100, width)
                        
                        block = intensity[row_start:row_end, col_start:col_end]
                        
                        # Exclude no data (255) from calculations
                        valid_pixels = block[block != 255]
                        
                        if valid_pixels.size > 0:
                            class_pixels = np.sum(valid_pixels == class_value)
                            grid[i, j] = class_pixels / valid_pixels.size
                        else:
                            grid[i, j] = -9999  # No valid data
                
                proportion_grids[class_name] = grid
                
                # Quick stats
                valid_cells = np.sum(grid >= 0)
                non_zero = np.sum(grid > 0)
                print(f" {non_zero}/{valid_cells} cells")
        
        # Save as multi-band GeoTIFF
        output_file = output_dir / f"lu_proportions_{country}_{year}_{version}.tif"
        
        print(f"\nSaving: {output_file.name}")
        with rasterio.open(output_file, 'w', **output_profile) as dst:
            for i, (class_name, grid) in enumerate(proportion_grids.items(), 1):
                dst.write(grid, i)
                dst.set_band_description(i, class_name)
        
        print(f"✅ Saved with {len(INTENSITY_CLASSES)} bands")

print("\n🎯 All proportions calculated and saved!")

### Single LU file

proportion Water include

In [ ]:
import rasterio
import numpy as np
from pathlib import Path

# ============================================
# Define intensity classes
# ============================================
INTENSITY_CLASSES = {
    'settlement_minimal': 10,
    'settlement_light': 11,
    'settlement_intense': 12,
    'plantation_minimal': 30,
    'plantation_light': 31,
    'plantation_intense': 32,
    'pasture_minimal': 60,
    'pasture_light': 61,
    'pasture_intense': 62,
    'crop_minimal': 70,
    'crop_light': 71,
    'crop_intense': 72,
    'sv_mature_minimal': 210,
    'sv_mature_light': 211,
    'sv_mature_intense': 212,
    'sv_intermediate_minimal': 220,
    'sv_intermediate_light': 221,
    'sv_intermediate_intense': 222,
    'sv_young_minimal': 230,
    'sv_young_light': 231,
    'sv_young_intense': 232,
    'sv_indeterminate_minimal': 240,
    'sv_indeterminate_light': 241,
    'sv_indeterminate_intense': 242,
}

# ============================================
# Choose what to process
# ============================================
country = 'dnk'  # 'dnk' or 'nld'
year = 2023      # 2018, 2021, or 2023

# ============================================
# Calculate proportions
# ============================================
print(f"Processing {country.upper()} {year}...")

# Load from new directory structure
input_file = Path(f"~/data/LEON_P5_BII/BII_LU_layer/Land_use_map/{version}/lu_intensity_{country}_{year}_{version}.tif").expanduser()
print(f"Loading: {input_file}")


with rasterio.open(input_file) as src:
    intensity = src.read(1)
    height, width = src.shape
    
    # 1km grid dimensions
    grid_height = height // 100
    grid_width = width // 100
    
    print(f"Original: {height} x {width} (10m)")
    print(f"Grid: {grid_height} x {grid_width} (1km)")
    
    # Store profile for later
    transform = src.transform * src.transform.scale(100, 100)
    output_profile = src.profile.copy()
    output_profile.update({
        'height': grid_height,
        'width': grid_width,
        'transform': transform,
        'dtype': 'float32'
    })
    
    # Calculate proportions for all classes
    proportion_grids = {}
    
    for class_name, class_value in INTENSITY_CLASSES.items():
        print(f"  Calculating {class_name}...")
        
        grid = np.zeros((grid_height, grid_width), dtype=np.float32)
        
        # Aggregate to 1km
        for i in range(grid_height):
            for j in range(grid_width):
                row_start = i * 100
                col_start = j * 100
                row_end = min(row_start + 100, height)
                col_end = min(col_start + 100, width)
                
                block = intensity[row_start:row_end, col_start:col_end]
                
                total_pixels = block.size
                class_pixels = np.sum(block == class_value)
                grid[i, j] = class_pixels / total_pixels if total_pixels > 0 else 0
        
        proportion_grids[class_name] = grid
        
        # Quick stats
        non_zero = np.sum(grid > 0)
        if non_zero > 0:
            print(f"    ✓ {non_zero} cells with this class")

print(f"\n✅ Proportions calculated and stored in 'proportion_grids' dict")

In [ ]:
# ============================================
# Save as multi-band GeoTIFF 
# ============================================
output_dir = Path(f"~/data/LEON_P5_BII/BII_LU_layer/Land_use_proportions/{version}").expanduser()
output_file = output_dir / f"LU_prop_{country}_{year}_{version}.tif"

# Update profile for multi-band
profile_multiband = output_profile.copy()
profile_multiband.update({'count': len(INTENSITY_CLASSES)})

print(f"\nSaving multi-band GeoTIFF for {country.upper()} {year}...")
with rasterio.open(output_file, 'w', **profile_multiband) as dst:
    for i, (class_name, grid) in enumerate(proportion_grids.items(), 1):
        dst.write(grid, i)
        dst.set_band_description(i, class_name)
        print(f"  Band {i:2d}: {class_name}")

print(f"✅ Saved: {output_file}")